# New York 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for New York, 2008 by merging the presidential primary and presidential general election results, and then derive summary stats (party totals). Note, there is no primary election dataset for New York 2008 so far, and there is a single general election dataset for Onondaga county.

**Output**: A single CSV where each row is a county and columns include:

- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_general_total` , `dem_general_total`, `lib_general_total`, `grn_general_total`, `psl_general_total`, `pop_general_total`, `swp_general_total`

**Last Updated**: 2025/10/22

## 0. Library Import

In [27]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Inputs & Parameters

Define raw file paths once here so the entire notebook is easy to rerun on another machine. If a path changes, we only update it here. We keep a single `OUTPUT_PATH` so all exports land in one known place.

In [39]:
# NY 2008 dataset path
# PRIMARY_PATH = r""
GENERAL_PATH = r"../../data/raw/2008/NY/20081104__ny__general__onondaga__precinct.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/NY/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

### b. General Election Dataset

In [61]:
# Load general data
general_df = pd.read_csv(GENERAL_PATH)
general_df.head(DISPLAY_ROWS)

,county,precinct,office,district,party,candidate,vote
0,Onondaga,W01 D01,Voters,NaN,NaN,Ballots Cast,398
1,Onondaga,W01 D02,Voters,NaN,NaN,Ballots Cast,305
2,Onondaga,W01 D03,Voters,NaN,NaN,Ballots Cast,375
3,Onondaga,W01 D04,Voters,NaN,NaN,Ballots Cast,389
4,Onondaga,W01 D05,Voters,NaN,NaN,Ballots Cast,275
5,Onondaga,W01 D06,Voters,NaN,NaN,Ballots Cast,265
6,Onondaga,W01 D07,Voters,NaN,NaN,Ballots Cast,300
7,Onondaga,W01 D08,Voters,NaN,NaN,Ballots Cast,304
8,Onondaga,W02 D01,Voters,NaN,NaN,Ballots Cast,330
9,Onondaga,W02 D02,Voters,NaN,NaN,Ballots Cast,426


In [62]:
# Different values in 'office' column
general_df["office"].value_counts()

office
President         5484
U S House         3199
State Assembly    3183
State Senate      2927
Voters             457
Name: count, dtype: int64

In [63]:
# Only keep rows where 'office' is 'President'
general_df = general_df[general_df["office"] == "President"]
general_df.head(DISPLAY_ROWS)

,county,precinct,office,district,party,candidate,vote
457,Onondaga,W01 D01,President,NaN,Dem,Barack Obama,233
458,Onondaga,W01 D02,President,NaN,Dem,Barack Obama,182
459,Onondaga,W01 D03,President,NaN,Dem,Barack Obama,239
460,Onondaga,W01 D04,President,NaN,Dem,Barack Obama,230
461,Onondaga,W01 D05,President,NaN,Dem,Barack Obama,165
462,Onondaga,W01 D06,President,NaN,Dem,Barack Obama,191
463,Onondaga,W01 D07,President,NaN,Dem,Barack Obama,177
464,Onondaga,W01 D08,President,NaN,Dem,Barack Obama,181
465,Onondaga,W02 D01,President,NaN,Dem,Barack Obama,203
466,Onondaga,W02 D02,President,NaN,Dem,Barack Obama,277


In [64]:
# General data shape when only considering President
general_df.shape

(5484, 7)

In [65]:
# Number of missing values in each column
general_df.isna().sum()

county          0
precinct        0
office          0
district     5484
party         914
candidate       0
vote            0
dtype: int64

In [66]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the "district" column since it's all missing values
general_df = general_df.drop(columns=["office", "district"]).reset_index(drop=True)
general_df.head(DISPLAY_ROWS)

,county,precinct,party,candidate,vote
0,Onondaga,W01 D01,Dem,Barack Obama,233
1,Onondaga,W01 D02,Dem,Barack Obama,182
2,Onondaga,W01 D03,Dem,Barack Obama,239
3,Onondaga,W01 D04,Dem,Barack Obama,230
4,Onondaga,W01 D05,Dem,Barack Obama,165
5,Onondaga,W01 D06,Dem,Barack Obama,191
6,Onondaga,W01 D07,Dem,Barack Obama,177
7,Onondaga,W01 D08,Dem,Barack Obama,181
8,Onondaga,W02 D01,Dem,Barack Obama,203
9,Onondaga,W02 D02,Dem,Barack Obama,277


Since we only have one county data, we can just groupby `party` and `candidate` to get the county vote counts.

In [67]:
# Calculate county vote counts
general_df = (
    general_df.groupby(["county", "candidate", "party"], as_index=False)["vote"].sum()
)

general_df.head(DISPLAY_ROWS)

,county,candidate,party,vote
0,Onondaga,Barack Obama,Dem,373728
1,Onondaga,Barack Obama,WF,14223
2,Onondaga,Bob Barr,LBT,3984
3,Onondaga,Cynthia McKinney,GRN,1197
4,Onondaga,Gloria LaRiva,SAL,222
5,Onondaga,John McCain,Con,19179
6,Onondaga,John McCain,Idc,14889
7,Onondaga,John McCain,Rep,220848
8,Onondaga,Ralph Nader,POP,5907
9,Onondaga,Roger Calero,SW,405


In [68]:
# Candidates in general_df
general_df["candidate"].value_counts()

candidate
John McCain         3
Barack Obama        2
Bob Barr            1
Cynthia McKinney    1
Gloria LaRiva       1
Ralph Nader         1
Roger Calero        1
Name: count, dtype: int64

There is something interesting with the resulted dataset after grouping: *New York fusion voting*. In NY, a candidate can appear on multiple party lines, and each line's votes are reported separatedly. In 2008 Onondaga:

* *Dem* = Democratic (Obama)

* *WF* = Working Families (cross-endorsed Obama, still counts for Obama)

* *Rep* = Republican (McCain)

* *Con* = Conservative (cross-endorsed McCain)

* *Idc* = Independence (cross-endorsed McCain)

* *LBT* = Libertarian (Bob Barr)

* *GRN* = Green (Cynthia McKinney)

* *SAL* = Party for Socialism & Liberation (Gloria La Riva)

* *POP* = “Populist” line (Ralph Nader’s ballot line in NY)

* *SW* = Socialist Workers (Roger Calero)

For analysis, we will sum across lines per candidate and collapse to major-party totals as follows:

In [69]:
# Candidate totals (sum across all party lines)
general_df = (
    general_df.groupby(['county', 'candidate'], as_index=False)['vote']
      .sum()
      .rename(columns={'vote': 'votes'})
      .assign(
        party=lambda d: d['candidate'].map(
            {
                'John McCain'      : 'rep', 
                'Barack Obama'     : 'dem',
                'Bob Barr'         : 'lib',
                'Cynthia McKinney' : 'grn',
                'Gloria LaRiva'    : 'psl',
                'Ralph Nader'      : 'pop',
                'Roger Calero'     : 'swp'       
            }
        ))
)

general_df.head(DISPLAY_ROWS)

,county,candidate,votes,party
0,Onondaga,Barack Obama,387951,dem
1,Onondaga,Bob Barr,3984,lib
2,Onondaga,Cynthia McKinney,1197,grn
3,Onondaga,Gloria LaRiva,222,psl
4,Onondaga,John McCain,254916,rep
5,Onondaga,Ralph Nader,5907,pop
6,Onondaga,Roger Calero,405,swp


In [70]:
# Data type of each column in general_df
general_df.dtypes

county       object
candidate    object
votes         int64
party        object
dtype: object

In [71]:
# Final look at the cleaned general_df
general_df.head(DISPLAY_ROWS)

,county,candidate,votes,party
0,Onondaga,Barack Obama,387951,dem
1,Onondaga,Bob Barr,3984,lib
2,Onondaga,Cynthia McKinney,1197,grn
3,Onondaga,Gloria LaRiva,222,psl
4,Onondaga,John McCain,254916,rep
5,Onondaga,Ralph Nader,5907,pop
6,Onondaga,Roger Calero,405,swp


In [72]:
# Shape after preprocessing
general_df.shape

(7, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: in this case, we lower everything so column names are stable with other dataframes
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [76]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names: Lowercase if not yet already
    """
    return(s.str.lower())

In [77]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [78]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [79]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_lib_BARR,gen_pop_NADER,gen_psl_LARIVA,gen_rep_MCCAIN,gen_swp_CALERO
0,Onondaga,387951,1197,3984,5907,222,254916,405


## 4. Adding Party Total Columns

Now, we will add party totals columns for general totals:

* `rep_general_total` = sum of all `gen_rep_*` columns
* `dem_general_total` = sum of all `gen_dem_*` columns
* `lib_general_total` = sum of all `gen_lib_*` columns
* `grn_general_total` = sum of all `gen_grn_*` columns
* `psl_general_total` = sum of all `gen_psl_*` columns
* `pop_general_total` = sum of all `gen_pop_*` columns
* `swp_general_total` = sum of all `gen_swp_*` columns

In [80]:
# Add party totals for general election
rep_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_rep")] 
dem_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_dem")]
lib_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_lib")]
grn_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_grn")]
psl_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_psl")]
pop_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_pop")]
swp_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_swp")]

general_pivot["rep_general_total"] = general_pivot[rep_general_cols].sum(axis=1) if rep_general_cols else 0
general_pivot["dem_general_total"] = general_pivot[dem_general_cols].sum(axis=1) if dem_general_cols else 0
general_pivot["lib_general_total"] = general_pivot[lib_general_cols].sum(axis=1) if lib_general_cols else 0
general_pivot["grn_general_total"] = general_pivot[grn_general_cols].sum(axis=1) if grn_general_cols else 0
general_pivot["psl_general_total"] = general_pivot[psl_general_cols].sum(axis=1) if psl_general_cols else 0
general_pivot["pop_general_total"] = general_pivot[pop_general_cols].sum(axis=1) if pop_general_cols else 0
general_pivot["swp_general_total"] = general_pivot[swp_general_cols].sum(axis=1) if swp_general_cols else 0

In [81]:
# Print out all the column names in the final dataframe
print("Final columns in the cleaned general dataframe:")
general_pivot.columns

Final columns in the cleaned general dataframe:


Index(['county', 'gen_dem_OBAMA', 'gen_grn_MCKINNEY', 'gen_lib_BARR',
       'gen_pop_NADER', 'gen_psl_LARIVA', 'gen_rep_MCCAIN', 'gen_swp_CALERO',
       'rep_general_total', 'dem_general_total', 'lib_general_total',
       'grn_general_total', 'psl_general_total', 'pop_general_total',
       'swp_general_total'],
      dtype='object')

In [82]:
# Preview the general_pivot dataframe with totals
general_pivot.head(DISPLAY_ROWS)

,county,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_lib_BARR,gen_pop_NADER,gen_psl_LARIVA,gen_rep_MCCAIN,gen_swp_CALERO,rep_general_total,dem_general_total,lib_general_total,grn_general_total,psl_general_total,pop_general_total,swp_general_total
0,Onondaga,387951,1197,3984,5907,222,254916,405,254916,387951,3984,1197,222,5907,405


Now, we save the cleaned dataframe into the processed directory.

In [84]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
general_pivot.to_csv(OUTPUT_PATH + "NY.csv", index=False)